# Importing libraries

In [1]:
import pandas as pd
import numpy as np

# Load the data

In [2]:
import os

# Raw dataset: UCI "Online Retail II" (two sheets: 2009-2010 and 2010-2011).
# Searches the project folder first, then common download locations.
DATA_PATH = next(
    p for p in [
        'online_retail_II.xlsx',
        'online_retail_II.csv',
        os.path.expanduser('~/Desktop/datasets-p2/online_retail_II.xlsx'),
    ]
    if os.path.exists(p)
)

if DATA_PATH.endswith('.xlsx'):
    xl = pd.ExcelFile(DATA_PATH)
    df = pd.concat([xl.parse(sheet) for sheet in xl.sheet_names], ignore_index=True)
else:
    df = pd.read_csv(DATA_PATH, encoding='utf-8-sig')

print('Loaded:', DATA_PATH)
df.shape

Loaded: /Users/rishabh/Desktop/datasets-p2/online_retail_II.xlsx


(1067371, 8)

# Look at the data

In [3]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


# Check for missing values

In [4]:
df.isnull().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

The 243007 missing customer ID are transactions that can't be attributed to any customers, so they're useless for RFM analysis. I will drop them. The missing product description doesn't matter for RFM since we're not using product descriptions. I'll leave these as-is.

# Drop rows with missing Customer ID

In [5]:
before = len(df)
df = df.dropna(subset=['Customer ID'])
print(f"Dropped {before - len(df)} rows with missing Customer ID")
print(df.shape)

Dropped 243007 rows with missing Customer ID
(824364, 8)


# Check for cancelled orders

Cancelled orders in this dataset have an Invoice number starting with "C"

In [6]:
cancelled = df[df['Invoice'].astype(str).str.startswith('C')]
print(f"Cancelled orders: {len(cancelled)}")
cancelled.head()

Cancelled orders: 18744


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia


# Remove cancelled orders and invalid quantity/price rows

In [7]:
before = len(df)
df = df[~df['Invoice'].astype(str).str.startswith('C')]
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]
print(f"Dropped {before - len(df)} cancelled/invalid rows")
print(df.shape)

Dropped 18815 cancelled/invalid rows
(805549, 8)


18,815 dropped because I included rows that had zero/negative price or quantity that weren't flagged as cancellations

# Parse the date column properly

In [8]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['InvoiceDate'].dtype

dtype('<M8[ns]')

# Remove duplicate rows

In [9]:
before = len(df)
df = df.drop_duplicates()
print(f"Dropped {before - len(df)} duplicate rows")
print(df.shape)

Dropped 26124 duplicate rows
(779425, 8)


# Add the features we'll need for RFM

In [10]:
# Total price per line item (Quantity x Price)
df['TotalPrice'] = df['Quantity'] * df['Price']

# Clean up Customer ID from a float to a clean integer string)
df['Customer ID'] = df['Customer ID'].astype(int).astype(str)

df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0


# Sanity check

In [11]:
print(f"Date range: {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")
print(f"Unique customers: {df['Customer ID'].nunique()}")
print(f"Countries: {df['Country'].nunique()}")

Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00
Unique customers: 5878
Countries: 41


# Save cleaned data

In [12]:
df.to_csv('cleaned_online_retail.csv', index=False)
print("Saved!")

Saved!
